In [ ]:
# Step 1: match_sop(Incident: diagnose root cause for window 2024-01-16 00:40:22 to 2024-01-16 01:10:22 UTC. Telemetry files available: metr
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.54)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.52)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.43)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2024-01-16 00:40:22'
end_time = '2024-01-16 01:10:22'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_summary = component_analyze(start_time, end_time)
answer = metric_evidence + ' ' + component_summary

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2024-01-16 00:40:22'
end_time = '2024-01-16 01:10:22'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_summary = component_analyze(start_time, end_time)
answer = metric_evidence + ' ' + component_summary
```

In [ ]:
# Step 3: run_sop
start_time = '2024-01-16 00:40:22'
end_time = '2024-01-16 01:10:22'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_summary = component_analyze(start_time, end_time)
answer = metric_evidence + ' ' + component_summary

```
[ANOMALY] emailservice latency-90: rise to 0.005199 at 2024-01-16 01:06:00 (median 0.0046, thr 0.004772, sev 49.0, n=2)
[ANOMALY] emailservice latency-50: rise to 0.003203 at 2024-01-16 01:06:00 (median 0.003, thr 0.003092, sev 45.0, n=2)
[ANOMALY] recommendationservice latency-50: rise to 0.03793 at 2024-01-16 01:05:00 (median 0.00808, thr 0.03751, sev 20.8, n=2)
[ANOMALY] currencyservice latency-90: rise to 0.1323 at 2024-01-16 00:48:00 (median 0.09774, thr 0.1068, sev 12.8, n=2)
[ANOMALY] recommendationservice latency-90: rise to 0.1926 at 2024-01-16 01:02:00 (median 0.02745, thr 0.1875, sev 6.2, n=2)
[ANOMALY] recommendationservice diskio: rise to 1.043e+10 at 2024-01-16 01:06:00 (median 1.044e+09, thr 1.013e+10, sev 6.1, n=2)
[ANOMALY] productcatalogservice cpu: drop to 2.936 at 2024-01-16 00:51:00 (median 3.431, thr 3.046, sev 5.9, n=2)
[ANOMALY] recommendationservice cpu: rise to 19.8 at 2024-01-16 01:01:00 (median 5.183, thr 19.63, sev 3.5, n=2) Component anomaly overview (ranked):
recommendationservice  metric_anoms=4 (top: latency-50 sev=20.8)  trace_anoms=9  log_anoms=1
productcatalogservice  metric_anoms=1 (top: cpu sev=5.9)  trace_anoms=9  log_anoms=0
checkoutservice  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=9  log_anoms=1
currencyservice  metric_anoms=1 (top: latency-90 sev=12.8)  trace_anoms=5  log_anoms=1
emailservice  metric_anoms=2 (top: latency-90 sev=49.0)  trace_anoms=3  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] emailservice latency-90: rise to 0.005199 at 2024-01-16 01:06:00 (median 0.0046, thr 0.004772, sev 49.0, n=2)

...[truncated])

```
ObAgent: The fault class is likely to be high disk I/O usage.

Analysis: recommendationservice shows a large diskio spike (rise to 1.043e+10 vs median 1.044e+09 — ~10x) concurrent with service-wide latency increases (email, recommendation, currency). CPU rises are smaller/isolated, not consistent with CPU saturation. This pattern points to I/O contention causing downstream latencies. To confirm, check host-level iowait and per-disk throughput/queue length (whether_is_abnormal_metric) and correlate with instance-level disk I/O.
JudgeAgent: Yes - Located component: recommendationservice; anomaly type: high disk I/O usage; evidence: recommendationservice diskio spike to 1.043e+10 at 2024-01-16 01:06:00 (median 1.044e+09, thr 1.013e+10), correlated with elevated latencies across services.
```